In [ ]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import random
import matplotlib.pyplot as plt
import time

# 🔥 🔥 🔥 新版本：可控密度的生成计划函数
def select_alpha_range(target_daily_tasks):
    """根据目标日均任务数，选择合适的退化率范围"""
    if target_daily_tasks <= 5:  # 稀疏
        return 0.004, 0.006
    elif 6 <= target_daily_tasks <= 12:  # 中等
        return 0.010, 0.013
    else:  # 13+ 密集
        return 0.020, 0.025


def generate_tasks_fixed_density(N, T, target_daily_tasks, seed=42):
    """根据目标任务密度生成维护计划"""
    random.seed(seed)

    caissons = list(range(N))
    initial_p_values = {c: round(random.uniform(3.2, 3.5), 2) for c in caissons}

    a, b = select_alpha_range(target_daily_tasks)
    alpha_values = {c: round(random.uniform(a, b), 6) for c in caissons}

    columns = ['INDEX', 'CSEM', 'INITIALP', 'OPERATION', 'DATE']
    data = []

    index = 0
    for caisson in caissons:
        initial_p = initial_p_values[caisson]
        current_p = initial_p
        alpha = alpha_values[caisson]
        total_days = 0

        while total_days < T:
            time_to_3_2 = int((current_p - 3.2) / alpha)
            if total_days + time_to_3_2 >= T:
                break

            data.append([index, caisson, initial_p, 'x', total_days + time_to_3_2])
            index += 1
            total_days += time_to_3_2 + 1
            current_p = 3.5

    maintenance_plan = pd.DataFrame(data, columns=columns)
    return maintenance_plan, alpha_values


# 🔥 🔥 🔥 计算维修周期（不变）
def calculate_maintenance_period_from_data(df, d_min_ratio=0.6, d_max_ratio=1.4):
    maintenance_periods = {}
    D_min_dict = {}
    D_max_dict = {}
    
    print("\n===== 从数据中提取维修周期 =====")
    print(f"{'Caisson':<10} {'间隔数':<10} {'间隔范围':<20} {'周期':<12} {'D_min':<10} {'D_max':<10}")
    print("-" * 80)
    
    for caisson_id in sorted(df['CSEM'].unique()):
        caisson_tasks = df[(df['CSEM'] == caisson_id) & (df['OPERATION'] == 'x')].sort_values('DATE')
        
        if len(caisson_tasks) < 2:
            default_period = PARAMS['window_size']
            maintenance_periods[caisson_id] = default_period
            D_min_dict[caisson_id] = int(np.ceil(default_period * d_min_ratio))
            D_max_dict[caisson_id] = int(np.ceil(default_period * d_max_ratio))
            print(f"{caisson_id:<10} ... {default_period:<12.2f} ...")  
            continue
        
        intervals = [caisson_tasks.iloc[i]['DATE'] - caisson_tasks.iloc[i-1]['DATE'] 
                    for i in range(1, len(caisson_tasks))]
        
        maintenance_period = np.mean(intervals)
        maintenance_periods[caisson_id] = maintenance_period
        D_min_dict[caisson_id] = int(np.ceil(maintenance_period * d_min_ratio))
        D_max_dict[caisson_id] = int(np.ceil(maintenance_period * d_max_ratio))
        
        interval_min, interval_max = min(intervals), max(intervals)

    
    print("===== 提取完成 =====\n")
    return maintenance_periods, D_min_dict, D_max_dict


# 🔥 🔥 🔥 Gurobi聚类函数（不变）
def clustering_CR(df_window, window_size, window_start, full_df=None, D_min_dict=None, D_max_dict=None):
    if df_window.empty or window_size < 1:
        print(f"warning: window {window_start} has no tasks")
        return df_window

    N = df_window.shape[0]
    mdl = gp.Model('Dynamic_Window_Clustering')
    mdl.setParam('OutputFlag', 0)

    data_pos_num = window_size + 1
    theta = mdl.addVars(N, data_pos_num, vtype=GRB.BINARY, name='theta')
    y = mdl.addVars(data_pos_num, vtype=GRB.BINARY, name='y')

    # 约束（完全不变）
    for n in range(N):
        mdl.addConstr(theta.sum(n, '*') == 1)
    for t in range(data_pos_num):
        mdl.addConstr(theta.sum('*', t) <= PARAMS['Cmax'])

    # 时间窗口约束
    a, b = [0] * N, [0] * N
    for n in range(N):
        local_time = int(df_window.iloc[n]['window_time'])
        a[n] = int(max(0, local_time - PARAMS['advanced']))
        b[n] = int(min(data_pos_num, local_time + int((3.2 - 3.0)/alpha_values[int(df_window.iloc[n]["CSEM"])])//2))

    for n in range(N):
        for t in range(data_pos_num):
            mdl.addConstr(t * theta[n, t] + a[n] * (1 - theta[n, t]) >= a[n])
            mdl.addConstr(t * theta[n, t] + b[n] * (1 - theta[n, t]) <= b[n])

    # 聚类激活
    M = PARAMS['Cmax']
    for t in range(data_pos_num):
        task_count = theta.sum('*', t)
        mdl.addConstr(task_count + M * (1 - y[t]) >= 1)
        mdl.addConstr(task_count <= M * y[t])

    # 目标函数：最大化利润
    setup_cost = PARAMS['Cs'] * (theta.sum() - y.sum())
    cost_matrix = {}
    for n in range(N):
        local_time_n = int(df_window.iloc[n]['window_time'])
        for t in range(data_pos_num):
            deviation = t - local_time_n
            cost_matrix[(n, t)] = PARAMS['Cbeta'] * max(0, deviation) + PARAMS['Calpha'] * max(0, -deviation)
    
    CP_total = gp.quicksum(cost_matrix[(n, t)] * theta[n, t] for n in range(N) for t in range(data_pos_num))
    mdl.setObjective(setup_cost - CP_total, GRB.MAXIMIZE)

    # 维护间隔约束
    if window_start > 0 and full_df is not None and D_min_dict is not None:
        for n in range(N):
            row = df_window.iloc[n]
            caisson = int(row['CSEM'])
            prev_rows = full_df[(full_df['CSEM'] == caisson) & (full_df['CENTER'] < row['DATE']) & 
                               (full_df['OPERATION'] == 'x')].sort_values('CENTER')
            if not prev_rows.empty:
                prev_center = int(prev_rows.iloc[-1]['CENTER'])
                t_now_local = gp.quicksum(t * theta[n, t] for t in range(data_pos_num))
                d_min_i = D_min_dict.get(caisson, PARAMS['window_size'] * 0.6)
                d_max_i = D_max_dict.get(caisson, PARAMS['window_size'] * 1.4)
                mdl.addConstr(t_now_local + window_start - prev_center >= d_min_i)
                mdl.addConstr(t_now_local + window_start - prev_center <= d_max_i)

    mdl.setParam('TimeLimit', 300)
    mdl.setParam('MIPGap', 0.05)
    mdl.setParam('OutputFlag', 1)

    print(f"Window {window_start}: N_tasks={N}")
    mdl.optimize()

    if mdl.status not in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL]:
        print(f"Failed to solve window {window_start}: {mdl.status}")
        return None
    
    centers = {}
    for n in range(N):
        for t in range(data_pos_num):
            if theta[n, t].x is not None and round(theta[n, t].x) == 1:
                centers[n] = t + window_start
                break

    index_n_list = df_window.INDEX.tolist()
    for n in range(len(index_n_list)):
        df_window.loc[df_window['INDEX'] == index_n_list[n], 'CENTER'] = centers.get(n, 0)

    return df_window


# 🔥 泄漏计算（不变）
def calculate_gas_leakage(df, T, use_center=True):
    total_leakage = 0
    for csem in df['CSEM'].unique():
        csem_data = df[df['CSEM'] == csem]
        x_operations = csem_data[csem_data['OPERATION'] == 'x'].sort_values('DATE' if not use_center else 'CENTER')
        if x_operations.empty:
            continue
        alpha = alpha_values[int(csem)]
        times = x_operations[('CENTER' if use_center else 'DATE')].values
        
        # 任务间泄漏
        for i in range(1, len(times)):
            center_diff = times[i] - times[i-1] - 1
            total_leakage += max(0, center_diff) * alpha * PARAMS['K']
        
        # 开头+结尾泄漏
        total_leakage += times[0] * alpha * PARAMS['K']
        total_leakage += max(0, T - (times[-1] + 1)) * alpha * PARAMS['K']
    return total_leakage


def update_secondary_tasks(main_df, clustered_df, window_start, window_end):
    for caisson in clustered_df["CSEM"].unique():
        tasks = main_df[(main_df['CSEM'] == caisson) & 
                       (main_df['DATE'].between(window_start, window_end)) & 
                       (main_df['OPERATION'] == 'x')].sort_values('DATE')
        if len(tasks) > 1:
            second_idx = tasks.index[1]
            center_time = clustered_df.loc[clustered_df['CSEM'] == caisson, 'CENTER'].iloc[0]
            alpha = alpha_values[int(caisson)]
            new_date = int(center_time + 1 + (3.5 - 3.2) / alpha)
            main_df.loc[second_idx, ['DATE', 'CENTER']] = new_date, new_date


# 🔥 🔥 🔥 主函数：统一seed管理
def run_clustering_analysis(params):
    """完整聚类分析 - 统一seed管理（params['random_seed']控制一切）"""
    global alpha_values, PARAMS
    PARAMS.update(params)
    
    # 🔥 🔥 🔥 统一seed：从params获取
    seed = params['random_seed']
    target_daily_tasks = params['target_daily_tasks']
    print(f"🔥 统一seed={seed}, 目标密度={target_daily_tasks}/天")
    
    # 🔥 生成计划（使用统一seed）
    df, alpha_values_dict = generate_tasks_fixed_density(
        N=params['P'], T=params['T'], target_daily_tasks=target_daily_tasks, seed=seed
    )
    alpha_values = alpha_values_dict
    
    print(f"生成 {len(df)} tasks")
    
    # 维修周期约束
    maintenance_periods, D_min_dict, D_max_dict = calculate_maintenance_period_from_data(df)
    
    # 聚类过程
    clustering_cpu_start = time.process_time()
    df_init = df.copy()
    df_history_list = [df_init]
    sliding_windows_starts, sliding_windows_ends = [], []
    df['CENTER'] = df['DATE']
    
    window_start = 0
    window_size = params['window_size']
    
    while window_start <= params['T']:
        window_end = min(window_start + window_size, params['T'])
        sliding_windows_starts.append(window_start)
        sliding_windows_ends.append(window_end)
        
        mask = (df['DATE'] >= window_start) & (df['DATE'] <= window_end) & (df['OPERATION'] == 'x')
        window_df = df.loc[mask].copy()
        if window_df.empty:
            break
            
        window_df['window_time'] = window_df['DATE'] - window_start
        first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])

        clustered_df = clustering_CR(first_tasks, window_size, window_start, df, D_min_dict, D_max_dict)
        
        if clustered_df is not None:
            center_map = clustered_df.set_index('INDEX')['CENTER']
            df.loc[df['INDEX'].isin(center_map.index), 'CENTER'] = df['INDEX'].map(center_map)
            update_secondary_tasks(df, clustered_df, window_start, window_end)
        
        second_dates = []
        if clustered_df is not None:
            for caisson in clustered_df["CSEM"].unique():
                tasks = df[(df['CSEM'] == caisson) & (df['OPERATION'] == 'x') &
                          (df['DATE'].between(window_start, window_end))].sort_values('DATE')
                if len(tasks) > 1:
                    second_dates.append(tasks.iloc[1]['DATE'])
        
        window_start = min(second_dates) if second_dates else window_start + window_size
        df_history_list.append(df.copy())
        
        if window_end >= params['T']:
            break
    
    # 未分配任务
    mask_remain = (df['OPERATION'] == "x") & (df['CENTER'] == df['DATE'])
    df.loc[mask_remain, 'CENTER'] = df.loc[mask_remain, 'DATE']

    total_clustering_cpu_time = time.process_time() - clustering_cpu_start

    # 泄漏计算
    leakage_before = calculate_gas_leakage(df_init, params['T'], use_center=False)
    leakage_after = calculate_gas_leakage(df, params['T'], use_center=True)
    reduction = leakage_before - leakage_after
    reduction_pct = (reduction / leakage_before * 100) if leakage_before > 0 else 0
    
    return {
        'final_df': df,
        'df_history_list': df_history_list,
        'sliding_windows_starts': sliding_windows_starts,
        'sliding_windows_ends': sliding_windows_ends,
        'alpha_values': list(alpha_values.values()),
        'maintenance_plan': df_init,
        'D_min_dict': D_min_dict,
        'D_max_dict': D_max_dict,
        'leakage_before': leakage_before,
        'leakage_after': leakage_after,
        'reduction': reduction,
        'reduction_pct': reduction_pct,
        'total_clustering_cpu_time': total_clustering_cpu_time,
        'used_seed': seed,  # 🔥 记录使用的seed
        'target_daily_tasks': target_daily_tasks
    }


def calculate_detailed_stats(result_dict, params):
    """
    计算详细统计 - 精确匹配你的CSV列
    """
    df_orig = result_dict['maintenance_plan'].copy()
    result_df = result_dict['final_df'].copy()
    
    # 只针对x任务
    df_x = df_orig[df_orig['OPERATION'] == 'x'].sort_values('INDEX').reset_index(drop=True)
    res_x = result_df[result_df['OPERATION'] == 'x'].sort_values('INDEX').reset_index(drop=True)
    
    # 🔥 🔥 🔥 核心计算
    time_diffs = res_x['CENTER'].values - df_x['DATE'].values
    
    # CP成本
    cp_cost = np.sum(np.where(time_diffs < 0, np.abs(time_diffs) * params['Calpha'], 
                             time_diffs * params['Cbeta']))
    
    # 聚类统计
    center_counts = res_x['CENTER'].value_counts()
    num_groups = len(center_counts)
    avg_tasks_per_group = center_counts.mean()
    
    # 时间偏移统计
    num_tasks = len(time_diffs)
    avg_shift_abs = np.mean(np.abs(time_diffs))
    max_shift_abs = np.max(np.abs(time_diffs))
    
    # 提前统计
    advanced_mask = time_diffs < 0
    avg_advance = np.mean(np.abs(time_diffs[advanced_mask])) if np.any(advanced_mask) else 0
    num_advanced = np.sum(advanced_mask)
    pct_advanced = (num_advanced / num_tasks * 100) if num_tasks > 0 else 0  # 🔥 改为百分比
    
    # 延迟统计
    delayed_mask = time_diffs > 0
    avg_delay = np.mean(time_diffs[delayed_mask]) if np.any(delayed_mask) else 0
    num_delayed = np.sum(delayed_mask)
    pct_delayed = (num_delayed / num_tasks * 100) if num_tasks > 0 else 0
    
    # 不变统计
    num_unchanged = np.sum(time_diffs == 0)
    pct_unchanged = (num_unchanged / num_tasks * 100) if num_tasks > 0 else 0
    
    # 🔥 🔥 🔥 CS成本（精确匹配）
    cs_cost_before = len(df_x) * params['Cs']
    cs_cost_after = num_groups * params['Cs']
    total_cost = cs_cost_after + cp_cost
    
    # 泄漏统计
    leakage_before = result_dict['leakage_before']
    leakage_after = result_dict['leakage_after']
    leakage_reduction = result_dict['reduction']
    leakage_reduction_pct = result_dict['reduction_pct']
    
    total_clustering_cpu_time = result_dict.get('total_clustering_cpu_time', 0)
    
    # 🔥 🔥 🔥 精确匹配你的CSV列！
    stats = {
        'seed': params['random_seed'],
        'nb_groups': num_groups,
        'Avg_tasks/group': round(avg_tasks_per_group, 3),
        'CS_cost_before': cs_cost_before,
        'CS_cost_after': cs_cost_after,
        'CP_cost': round(cp_cost, 3),
        'Total_cost': round(total_cost, 3),
        'Avg_shift_(abs)': round(avg_shift_abs, 3),
        'Max_shift_(abs)': round(max_shift_abs, 3),
        'Avg_advance_(days)': round(avg_advance, 3),
        'Avg_delay_(days)': round(avg_delay, 3),
        'pct_advanced_%': round(pct_advanced, 3),  # 🔥 百分比
        'pct_delayed_%': round(pct_delayed, 3),
        'pct_unchanged_%': round(pct_unchanged, 3),
        'leakage_before': round(leakage_before, 3),
        'leakage_after': round(leakage_after, 3),
        'leakage_reduction': round(leakage_reduction, 3),
        'leakage_reduction_pct': round(leakage_reduction_pct, 3),
        'total_clustering_cpu_s': round(total_clustering_cpu_time, 3)
    }
    
    return stats



def run_analysis_with_csv_export(params_template, seeds, output_filename='detailed_results.csv'):
    """批量运行 - 精确CSV列顺序"""
    all_results = []
    
    for seed in seeds:
        params = params_template.copy()
        params['random_seed'] = seed
        
        print(f"\n🔄 Processing seed {seed}...")
        result_dict = run_clustering_analysis(params)
        stats = calculate_detailed_stats(result_dict, params)
        all_results.append(stats)
    
    results_df = pd.DataFrame(all_results)
    
    # 🔥 🔥 🔥 精确匹配你的列顺序！
    exact_columns = [
        'seed','nb_groups','Avg_tasks/group','CS_cost_before','CS_cost_after',
        'CP_cost','Total_cost','Avg_shift_(abs)','Max_shift_(abs)',
        'Avg_advance_(days)','Avg_delay_(days)','pct_advanced_%','pct_delayed_%',
        'pct_unchanged_%','leakage_before','leakage_after','leakage_reduction',
        'leakage_reduction_pct','total_clustering_cpu_s'
    ]
    
    results_df = results_df[exact_columns]  # 强制精确顺序
    
    results_df.to_csv(output_filename, index=False, float_format='%.3f')
    print(f"\n📁 CSV已保存: {output_filename}")
    print(f"📊 列顺序: {list(results_df.columns)}")
    
    return results_df


# PARAMS_TEMPLATE = {
#     'P': 100, 'T': 180, 'Cmax': 20, 'window_size': 10, 'advanced': 3,
#     'K': 5.993, 'Calpha': 0.01, 'Cbeta': 0.02, 'Cs': 100,
#     'target_daily_tasks': 20, 'random_seed': 1
# }

# seeds_to_test = [1]
# df_results = run_analysis_with_csv_export(PARAMS_TEMPLATE, seeds_to_test, 'COST.csv')

# print("\n✅ CSV列精确匹配：")
# print(df_results.columns.tolist())
# print("\n📋 数据预览：")
# print(df_results.round(3).head())






# 不同任务密度待测试值
task_densities = [1, 10, 20, 50, 100]

# 固定模型其他参数
BASE_PARAMS = {
    'P': 100,
    'T': 180,
    'Cmax': 20,
    'window_size': 20,
    'advanced': 3,
    'K': 5.993,
    'Calpha': 5,
    'Cbeta': 10,
    'Cs': 100
}

seeds_to_test = [1]   # 可设置多个例如 range(1, 6)

all_results = []  # 用来收集所有实验结果

for td in task_densities:
    print(f"\n============================")
    print(f"🔍 Running for target_daily_tasks = {td}")
    print(f"============================\n")

    # 更新参数
    PARAMS = BASE_PARAMS.copy()
    PARAMS['target_daily_tasks'] = td

    # 定义输出文件名
    filename = f"results_p{PARAMS['P']}t{PARAMS['T']}_td{td}.csv"

    # 运行分析并导出结果
    result_df = run_analysis_with_csv_export(PARAMS, seeds_to_test, output_filename=filename)
    result_df['target_daily_tasks'] = td  # 加入密度这一列
    all_results.append(result_df)

# 汇总所有结果成一个大的DataFrame
summary_df = pd.concat(all_results, ignore_index=True)
summary_df.to_csv("summary_all_task_densities_COST.csv", index=False)
print("\n✅ 所有密度实验已完成并保存至 summary_all_task_densities_COST.csv")

# 预览
print(summary_df.head())



🔍 Running for target_daily_tasks = 1


🔄 Processing seed 1...
🔥 统一seed=1, 目标密度=1/天
生成 293 tasks

===== 从数据中提取维修周期 =====
Caisson    间隔数        间隔范围                 周期           D_min      D_max     
--------------------------------------------------------------------------------
===== 提取完成 =====

Set parameter OutputFlag to value 1
Window 0: N_tasks=28
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 1267 rows, 609 columns and 3537 nonzeros
Model fingerprint: 0xbb39a31b
Variable types: 0 continuous, 609 integer (609 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [5e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]
Found heuristic solution: objective 5.0000000
Presolve removed 1197 rows and 187 columns
Presolve time: 0.00s
Presolved: 70 rows, 422 columns, 1245 nonzeros
Variable type

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_52759/1541592730.py:259: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_52759/1541592730.py:259: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals

     0     0 3342.78190    0   91 3110.00000 3342.78190  7.48%     -    0s
     0     0 3342.78190    0  103 3110.00000 3342.78190  7.48%     -    0s
H    0     0                    3120.0000000 3320.12835  6.41%     -    0s
H    0     0                    3155.0000000 3320.12835  5.23%     -    0s
     0     0 3320.12835    0   38 3155.00000 3320.12835  5.23%     -    0s
H    0     0                    3160.0000000 3320.12835  5.07%     -    0s
     0     0 3318.16138    0   42 3160.00000 3318.16138  5.01%     -    0s

Cutting planes:
  Cover: 34
  Clique: 58
  MIR: 29
  StrongCG: 2
  Zero half: 1

Explored 1 nodes (196 simplex iterations) in 0.02 seconds (0.02 work units)
Thread count was 12 (of 12 available processors)

Solution count 10: 3160 3155 3120 ... 470

Optimal solution found (tolerance 5.00e-02)
Best objective 3.160000000000e+03, best bound 3.315000000000e+03, gap 4.9051%
Set parameter OutputFlag to value 1
Window 100: N_tasks=36
Gurobi Optimizer version 10.0.1 build v10.0

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_52759/1541592730.py:259: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])
/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_52759/1541592730.py:259: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=Fals


📁 CSV已保存: results_p100t180_td1.csv
📊 列顺序: ['seed', 'nb_groups', 'Avg_tasks/group', 'CS_cost_before', 'CS_cost_after', 'CP_cost', 'Total_cost', 'Avg_shift_(abs)', 'Max_shift_(abs)', 'Avg_advance_(days)', 'Avg_delay_(days)', 'pct_advanced_%', 'pct_delayed_%', 'pct_unchanged_%', 'leakage_before', 'leakage_after', 'leakage_reduction', 'leakage_reduction_pct', 'total_clustering_cpu_s']

🔍 Running for target_daily_tasks = 10


🔄 Processing seed 1...
🔥 统一seed=1, 目标密度=10/天
生成 671 tasks

===== 从数据中提取维修周期 =====
Caisson    间隔数        间隔范围                 周期           D_min      D_max     
--------------------------------------------------------------------------------
===== 提取完成 =====

Set parameter OutputFlag to value 1
Window 0: N_tasks=80
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 3503 rows, 1701 columns and 9997 nonzeros
Model fingerprint: 0x4f22a8

/var/folders/08/0s62qkn131s4xxtxmk7knl840000gp/T/ipykernel_52759/1541592730.py:259: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_tasks = window_df.groupby('CSEM', group_keys=False).apply(lambda x: x.sort_values('DATE').iloc[0])


KeyboardInterrupt: 